## LibreLane Colab — RISC-V Single-Cycle Processor (RTL2GDS)

This Google Colab notebook will:
* Install LibreLane and its dependencies.
* Run a **single-cycle RV32I RISC-V processor** (`top_module`) through the full
  RTL-to-GDSII flow, targeting the open source
  [sky130 PDK](https://github.com/google/skywater-pdk/) by Google and SkyWater.

The design is the single-cycle core from this repository: program counter,
instruction/data memory, register file, immediate generator, control unit,
ALU and ALU control.


In [ ]:
# @title Setup Nix {display-mode: "form"}
# @markdown <img src="https://raw.githubusercontent.com/NixOS/nixos-artwork/51a27e4a011e95cb559e37d32c44cf89b50f5154/logo/nix-snowflake-colours.svg" width="32"/>
# @markdown
# @markdown Nix is a package manager with an emphasis on reproducible builds,
# @markdown and it is the primary method for installing LibreLane.
# @markdown
# @markdown This step installs the Nix package manager and enables the
# @markdown FOSSi Foundation Nix Cache.
# @markdown
# @markdown If you're not in a Colab, this just sets the environment variables.
# @markdown You will need to install Nix and enable flakes on your own following
# @markdown [this guide](https://librelane.readthedocs.io/en/stable/getting_started/common/nix_installation/index.html).
import os
from pathlib import Path
import subprocess
import sys
import shutil
import tempfile

os.environ["LOCALE_ARCHIVE"] = "/usr/lib/locale/locale-archive"

if "google.colab" in sys.modules:
    if shutil.which("nix-env") is None:
        with tempfile.TemporaryDirectory() as d:
            d = Path(d)
            installer_path = d / "nix"
            !curl --proto '=https' --tlsv1.2 -sSf -L https://install.determinate.systems/nix > {installer_path}
            with subprocess.Popen(
                [
                    "bash",
                    installer_path,
                    "install",
                    "--prefer-upstream-nix",
                    "--no-confirm",
                    "--extra-conf",
                    "extra-substituters = https://nix-cache.fossi-foundation.org\nextra-trusted-public-keys = nix-cache.fossi-foundation.org:3+K59iFwXqKsL7BNu6Guy0v+uTlwsxYQxjspXzqLYQs=\n",
                ],
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                encoding="utf8",
            ) as p:
                for line in p.stdout:
                    print(line, end="")
else:
    if shutil.which("nix-env") is None:
        raise RuntimeError("Nix is not installed!")

os.environ["PATH"] = f"/nix/var/nix/profiles/default/bin/:{os.getenv('PATH')}"

In [ ]:
# @title Get LibreLane {display-mode: "form"}
# @markdown Click the ▷ button to download and install LibreLane.
# @markdown
# @markdown This will install LibreLane's tool dependencies using Nix,
# @markdown and LibreLane itself using PIP.
# @markdown
# @markdown Note that `python3-tk` may need to be installed using your OS's
# @markdown package manager.
import os
import yaml
import subprocess
import IPython

librelane_version = "latest"  # @param {key:"LibreLane Version", type:"string"}

if librelane_version == "latest":
    librelane_version = "main"

pdk_root = "~/.ciel"  # @param {key:"PDK Root", type:"string"}

pdk_root = os.path.expanduser(pdk_root)

pdk = "sky130"  # @param {key:"PDK (without the variant)", type:"string"}

librelane_ipynb_path = os.path.join(os.getcwd(), "librelane_ipynb")

display(IPython.display.HTML("<h3>Downloading LibreLane…</a>"))


TESTING_LOCALLY = False
!rm -rf {librelane_ipynb_path}
!mkdir -p {librelane_ipynb_path}
if TESTING_LOCALLY:
    !ln -s {os.getcwd()} {librelane_ipynb_path}
else:
    !curl -L "https://github.com/librelane/librelane/tarball/{librelane_version}" | tar -xzC {librelane_ipynb_path} --strip-components 1

try:
    import tkinter
except ImportError:
    if "google.colab" in sys.modules:
        !sudo apt-get install python-tk

try:
    import tkinter
except ImportError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to import the <code>tkinter</code> library for Python, which is required to load PDK configuration values. Make sure <code>python3-tk</code> or equivalent is installed on your system.</a>'
        )
    )
    raise e from None


display(IPython.display.HTML("<h3>Downloading LibreLane's dependencies…</a>"))
try:
    with subprocess.Popen(
        [
            "nix",
            "profile",
            "install",
            ".#colab-env",
        ],
        cwd=librelane_ipynb_path,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        encoding="utf8",
    ) as p:
        for line in p.stdout:
            print(line, end="")
except subprocess.CalledProcessError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to install binary dependencies using Nix…</h3>'
        )
    )

display(IPython.display.HTML("<h3>Downloading Python dependencies using PIP…</a>"))
try:
    subprocess.check_call(
        ["pip3", "install", "."],
        cwd=librelane_ipynb_path,
    )
except subprocess.CalledProcessError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to install Python dependencies using PIP…</h3>'
        )
    )
    raise e from None

display(IPython.display.HTML("<h3>Downloading PDK…</a>"))
import ciel
from ciel.source import StaticWebDataSource

with open(
    os.path.join(librelane_ipynb_path, "librelane", "pdk_hashes.yaml"), "r"
) as file:
    pdk_hashes = yaml.safe_load(file)

ciel.enable(
    ciel.get_ciel_home(pdk_root),
    pdk,
    pdk_hashes[pdk],
    data_source=StaticWebDataSource("https://fossi-foundation.github.io/ciel-releases"),
)

sys.path.insert(0, librelane_ipynb_path)
display(IPython.display.HTML("<h3>⭕️ Done.</a>"))

import logging

# Remove the stupid default colab logging handler
logging.getLogger().handlers.clear()

In [ ]:
import librelane

print(librelane.__version__)

### Creating the design

The cell below writes the complete RISC-V core (all modules in a single Verilog
file) and the instruction-memory contents (`instrMem.hex`).

> **Note:** the original `top_module` only had `clk` / `rst` and **no outputs**,
> so synthesis would optimize the entire datapath away. Two debug outputs
> (`pc_debug`, `result_debug`) are exposed here so the logic is preserved through
> the flow — they do not change the processor's behaviour.


In [ ]:
%%writefile top_module.v
// =====================================================================
//  Single-cycle RV32I processor - consolidated for the LibreLane flow.
//  NOTE: top_module exposes pc_debug / result_debug so that synthesis
//  keeps the whole datapath (the original core had no outputs and would
//  be optimized away entirely).
// =====================================================================

module top_module(
    input         clk,
    input         rst,
    output [31:0] pc_debug,
    output [31:0] result_debug
);

wire [31:0] pc;
wire [31:0] pc_next;
wire [31:0] pc4;
wire [31:0] pc_target;

wire [31:0] instruction;

wire [31:0] rd1;
wire [31:0] rd2;

wire [31:0] imm_ext;

wire [31:0] src_b;
wire [31:0] alu_result;
wire        zero;

wire [31:0] read_data;
wire [31:0] result;

wire pc_src;
wire mem_write;
wire alu_src;
wire reg_write;

wire [1:0] result_src;
wire [1:0] imm_src;
wire [1:0] alu_op;
wire [2:0] alu_control;

// Debug outputs keep the design from being optimized away.
assign pc_debug     = pc;
assign result_debug = result;

// PROGRAM COUNTER
program_counter DUT_PC(
    .clk(clk),
    .rst(rst),
    .pc_next(pc_next),
    .pc_out(pc)
);

// INSTRUCTION MEMORY
instruction_memory DUT_IMEM(
    .clk(clk),
    .A(pc),
    .RD(instruction)
);

// PC + 4
adder DUT_PC4(
    .a(pc),
    .b(32'd4),
    .result(pc4)
);

// PC + INMEDIATO
adder DUT_PCTARGET(
    .a(pc),
    .b(imm_ext),
    .result(pc_target)
);

// MUX PARA SIGUIENTE PC
multiplexor DUT_MUX_PC(
    .A(pc4),
    .B(pc_target),
    .C(32'b0),
    .D(32'b0),
    .Sel({1'b0, pc_src}),
    .Y(pc_next)
);

// REGISTER FILE
reg_file DUT_REGFILE(
    .clk(clk),
    .WE3(reg_write),
    .A1(instruction[19:15]),
    .A2(instruction[24:20]),
    .A3(instruction[11:7]),
    .WD3(result),
    .RD1(rd1),
    .RD2(rd2)
);

// EXTEND / IMM GEN
imm_gen DUT_IMMGEN(
    .instruction(instruction),
    .imm_src(imm_src),
    .imm_ext(imm_ext)
);

// CONTROL UNIT
control_unit DUT_CONTROL(
    .op(instruction[6:0]),
    .zero(zero),
    .pc_src(pc_src),
    .mem_write(mem_write),
    .alu_src(alu_src),
    .reg_write(reg_write),
    .result_src(result_src),
    .imm_src(imm_src),
    .alu_op(alu_op)
);

// ALU CONTROL
alu_control DUT_ALUCONTROL(
    .alu_op(alu_op),
    .funct3(instruction[14:12]),
    .funct7(instruction[30]),
    .alu_control(alu_control)
);

// MUX PARA ENTRADA B DE ALU
multiplexor DUT_MUX_ALU_B(
    .A(rd2),
    .B(imm_ext),
    .C(32'b0),
    .D(32'b0),
    .Sel({1'b0, alu_src}),
    .Y(src_b)
);

// ALU
alu DUT_ALU(
    .a(rd1),
    .b(src_b),
    .alu_control(alu_control),
    .result(alu_result),
    .zero(zero)
);

// DATA MEMORY
data_memory DUT_DATAMEM(
    .clk(clk),
    .WE(mem_write),
    .A(alu_result),
    .WD(rd2),
    .RD(read_data)
);

// MUX RESULT
multiplexor DUT_MUX_RESULT(
    .A(alu_result),
    .B(read_data),
    .C(pc4),
    .D(32'b0),
    .Sel(result_src),
    .Y(result)
);

endmodule


module program_counter(
    input clk,
    input rst,
    input [31:0] pc_next,
    output reg [31:0] pc_out
);
always @(posedge clk or posedge rst) begin
    if (rst)
        pc_out <= 32'd0;
    else
        pc_out <= pc_next;
end
endmodule


module instruction_memory(
    input clk,
    input [31:0] A,
    output reg [31:0] RD
);
    reg [31:0] instr_mem [0:2];
    initial begin
        $readmemh("instrMem.hex", instr_mem);
    end
    always @(posedge clk)
        RD <= instr_mem[A[31:2]];
endmodule


module adder(
    input  [31:0] a, b,
    output [31:0] result
);
    assign result = a + b;
endmodule


module multiplexor(
    input  [31:0] A,
    input  [31:0] B,
    input  [31:0] C,
    input  [31:0] D,
    input  [1:0]  Sel,
    output reg [31:0] Y
);
always @(*) begin
    case (Sel)
        2'b00: Y = A;
        2'b01: Y = B;
        2'b10: Y = C;
        2'b11: Y = D;
        default: Y = 32'b0;
    endcase
end
endmodule


module reg_file(
    input clk, WE3,
    input  [4:0]  A1, A2, A3,
    input  [31:0] WD3,
    output reg [31:0] RD1, RD2
);
    reg [31:0] REG [31:0];
    always @(*) begin
        if (A1 == 0) RD1 = 0; else RD1 = REG[A1];
        if (A2 == 0) RD2 = 0; else RD2 = REG[A2];
    end
    always @(posedge clk) begin
        if (WE3 == 1 && A3 != 0)
            REG[A3] <= WD3;
    end
endmodule


module imm_gen(
    input  [31:0] instruction,
    input  [1:0]  imm_src,
    output reg [31:0] imm_ext
);
always @(*) begin
    case (imm_src)
        2'b00: imm_ext = {{20{instruction[31]}}, instruction[31:20]};
        2'b01: imm_ext = {{20{instruction[31]}}, instruction[31:25], instruction[11:7]};
        2'b10: imm_ext = {{20{instruction[31]}}, instruction[7], instruction[30:25], instruction[11:8], 1'b0};
        2'b11: imm_ext = {{12{instruction[31]}}, instruction[19:12], instruction[20], instruction[30:21], 1'b0};
        default: imm_ext = 32'b0;
    endcase
end
endmodule


module control_unit(
    input  [6:0] op,
    input        zero,
    output reg       pc_src,
    output reg       mem_write,
    output reg       alu_src,
    output reg       reg_write,
    output reg [1:0] result_src,
    output reg [1:0] imm_src,
    output reg [1:0] alu_op
);
always @(*) begin
    pc_src     = 0;
    mem_write  = 0;
    alu_src    = 0;
    reg_write  = 0;
    result_src = 2'b00;
    imm_src    = 2'b00;
    alu_op     = 2'b00;
    case (op)
        7'b0000011: begin // lw
            reg_write = 1; alu_src = 1; result_src = 2'b01; imm_src = 2'b00; alu_op = 2'b00;
        end
        7'b0100011: begin // sw
            mem_write = 1; alu_src = 1; imm_src = 2'b01; alu_op = 2'b00;
        end
        7'b0110011: begin // R-type
            reg_write = 1; alu_src = 0; result_src = 2'b00; alu_op = 2'b10;
        end
        7'b0010011: begin // I-type addi
            reg_write = 1; alu_src = 1; result_src = 2'b00; imm_src = 2'b00; alu_op = 2'b10;
        end
        7'b1100011: begin // beq
            pc_src = zero; alu_src = 0; imm_src = 2'b10; alu_op = 2'b01;
        end
        7'b1101111: begin // jal
            pc_src = 1; reg_write = 1; result_src = 2'b10; imm_src = 2'b11; alu_op = 2'b00;
        end
        default: begin
            pc_src = 0; mem_write = 0; alu_src = 0; reg_write = 0;
            result_src = 2'b00; imm_src = 2'b00; alu_op = 2'b00;
        end
    endcase
end
endmodule


module alu_control(
    input  [1:0] alu_op,
    input  [2:0] funct3,
    input        funct7,
    output reg [2:0] alu_control
);
always @(*) begin
    case (alu_op)
        2'b00: alu_control = 3'b000; // ADD (lw/sw)
        2'b01: alu_control = 3'b001; // SUB (branch)
        2'b10: begin
            case (funct3)
                3'b000: alu_control = (funct7 == 1'b0) ? 3'b000 : 3'b001; // ADD/SUB
                3'b111: alu_control = 3'b010; // AND
                3'b110: alu_control = 3'b011; // OR
                3'b010: alu_control = 3'b101; // SLT
                default: alu_control = 3'b000;
            endcase
        end
        default: alu_control = 3'b000;
    endcase
end
endmodule


module alu(
    input  [31:0] a,
    input  [31:0] b,
    input  [2:0]  alu_control,
    output reg [31:0] result,
    output        zero
);
always @(*) begin
    case (alu_control)
        3'b000: result = a + b;                       // ADD
        3'b001: result = a - b;                       // SUB
        3'b010: result = a & b;                       // AND
        3'b011: result = a | b;                       // OR
        3'b101: result = (a < b) ? 32'd1 : 32'd0;     // SLT
        default: result = 32'd0;
    endcase
end
assign zero = (result == 32'd0);
endmodule


module data_memory(
    input clk,
    input WE,
    input  [31:0] A,
    input  [31:0] WD,
    output [31:0] RD
);
    reg [31:0] data_mem [0:15]; // reduced from [0:255] for a fast Colab P&R run
    assign RD = data_mem[A[31:2]];
    always @(posedge clk) begin
        if (WE)
            data_mem[A[31:2]] <= WD;
    end
endmodule


In [ ]:
%%writefile instrMem.hex
00500293
00a00313
00628533


### Setting up the configuration

LibreLane requries you to configure any Flow before using it. This is done using
the `config` module.

For colaboratories, REPLs and other interactive environments where there is no
concrete Flow object, the Configuration may be initialized using `Config.interactive`,
which will automatically propagate the configuration to any future steps.

You can find the documentation for `Config.interactive` [here](https://librelane.readthedocs.io/en/latest/reference/api/config/index.html#librelane.config.Config.interactive).



In [ ]:
from librelane.config import Config

Config.interactive(
    "top_module",
    PDK="sky130A",
    CLOCK_PORT="clk",
    CLOCK_NET="clk",
    CLOCK_PERIOD=20,
    PRIMARY_GDSII_STREAMOUT_TOOL="klayout",
)


### Running implementation steps

There are two ways to obtain LibreLane's built-in implementation steps:

* via directly importing from the `steps` module using its category:
    * `from librelane.steps import Yosys` then `Synthesis = Yosys.Synthesis`
* by using the step's id from the registry:
    * `from librelane.steps import Step` then `Synthesis = Step.factory.get("Yosys.Synthesis")`

You can find a full list of included steps here: https://librelane.readthedocs.io/en/latest/reference/step_config_vars.html

In [ ]:
from librelane.steps import Step

* First, get the step (and display its help)...

In [ ]:
Synthesis = Step.factory.get("Yosys.Synthesis")

Synthesis.display_help()

* Then run it. Note you can pass step-specific configs using Python keyword
  arguments.

### Synthesis

We need to start by converting our high-level Verilog to one that just shows
the connections between small silicon patterns called "standard cells" in process
called Synthesis. We can do this by passing the Verilog files as a configuration
variable to `Yosys.Synthesis` as follows, then running it.

As this is the first step, we need to create an empty state and pass it to it.

In [ ]:
from librelane.state import State

synthesis = Synthesis(
    VERILOG_FILES=["./top_module.v"],
    state_in=State(),
)
synthesis.start()


In [ ]:
display(synthesis)

### Floorplanning

Floorplanning does two things:

* Determines the dimensions of the final chip.
* Creates the "cell placement grid" which placed cells must be aligned to.
    * Each cell in the grid is called a "site." Cells can occupy multiple
      sites, with the overwhelming majority of cells occupying multiple sites
      by width, and some standard cell libraries supporting varying heights as well.

> Don't forget- you may call `display_help()` on any Step class to get a full
> list of configuration variables.


In [ ]:
Floorplan = Step.factory.get("OpenROAD.Floorplan")

floorplan = Floorplan(state_in=synthesis.state_out)
floorplan.start()

In [ ]:
display(floorplan)

### Tap/Endcap Cell Insertion

This places two kinds of cells on the floorplan:

* End cap/boundary cells: Added at the beginning and end of each row. True to
  their name, they "cap off" the core area of a design.
* Tap cells: Placed in a polka dot-ish fashion across the rows. Tap cells
  connect VDD to the nwell and the psubstrate to VSS, which the majority of cells
  do not do themselves to save area- but if you go long enough without one such
  connection you end up with the cell "latching-up"; i.e.; refusing to switch
  back to LO from HI.

  There is a maximum distance between tap cells enforced as part of every
  foundry process.

In [ ]:
TapEndcapInsertion = Step.factory.get("OpenROAD.TapEndcapInsertion")

tdi = TapEndcapInsertion(state_in=floorplan.state_out)
tdi.start()

In [ ]:
!find /content -name "config.json"


In [ ]:
display(tdi)

### I/O Placement

This places metal pins at the edges of the design corresponding to the top level
inputs and outputs for your design. These pins act as the interface with other
designs when you integrate it with other designs.

In [ ]:
IOPlacement = Step.factory.get("OpenROAD.IOPlacement")

ioplace = IOPlacement(state_in=tdi.state_out)
ioplace.start()

In [ ]:
display(ioplace)

### Generating the Power Distribution Network (PDN)

This creates the power distribution network for your design, which is essentially
a plaid pattern of horizontal and vertical "straps" across the design that is
then connected to the rails' VDD and VSS (via the tap cells.)

You can find an explanation of how the power distribution network works at this
link: https://librelane.readthedocs.io/en/latest/usage/hardening_macros.html#pdn-generation

We typically don't need to mess with the PDN much, so here we just use the PDK's default power grid. (The original SPM tutorial forced a much denser grid, but OpenROAD rejects that pitch on this metal stack.)

In [ ]:
GeneratePDN = Step.factory.get("OpenROAD.GeneratePDN")

# Use the PDK's default power-grid geometry.
# The original SPM tutorial forced a very dense grid (30um pitch), which
# OpenROAD rejects on this metal stack:
#   [PDN-0175] Pitch 30.0000 is too small, must be at least 53.4000
pdn = GeneratePDN(state_in=ioplace.state_out)
pdn.start()


In [ ]:
display(pdn)

### Global Placement

Global Placement is deciding on a fuzzy, non-final location for each of the cells,
with the aim of minimizing the distance between cells that are connected
together (more specifically, the total length of the not-yet-created wires that
will connect them).

As you will see in the `.display()` in the second cell below, the placement is
considered "illegal", i.e., not properly aligned with the cell placement grid.
This is addressed by "Detailed Placement", also referred to as "placement
legalization", which is the next step.

In [ ]:
GlobalPlacement = Step.factory.get("OpenROAD.GlobalPlacement")

gpl = GlobalPlacement(state_in=pdn.state_out)
gpl.start()

In [ ]:
display(gpl)

### Detailed Placement

This aligns the fuzzy placement from before with the grid, "legalizing" it.

In [ ]:
DetailedPlacement = Step.factory.get("OpenROAD.DetailedPlacement")

dpl = DetailedPlacement(state_in=gpl.state_out)
dpl.start()

In [ ]:
display(dpl)

### Clock Tree Synthesis (CTS)

With the cells now having a final placement, we can go ahead and create what
is known as the clock tree, i.e., the hierarchical set of buffers used
for clock signal to minimize what is known as "clock skew"- variable delay
of the clock cycle from register to register because of factors such as metal
wire length, clock load (number of gates connected to the same clock buffer,)
et cetera.

The CTS step creates the cells and places the between the gaps in the detailed
placement above.

In [ ]:
CTS = Step.factory.get("OpenROAD.CTS")

cts = CTS(state_in=dpl.state_out)
cts.start()

In [ ]:
display(cts)

### Global Routing

Global routing "plans" the routes the wires between two gates (or gates and
I/O pins/the PDN) will take. The results of global routing (which are called
"routing guides") are stored in internal data structures and have no effect on
the actual design, so there is no `display()` statement.

In [ ]:
GlobalRouting = Step.factory.get("OpenROAD.GlobalRouting")

grt = GlobalRouting(state_in=cts.state_out)
grt.start()

### Detailed Routing

Detailed routing uses the guides from Global Routing to actually create wires
on the metal layers and connect the gates, making the connections finally physical.

This is typically the longest step in the flow.

In [ ]:
DetailedRouting = Step.factory.get("OpenROAD.DetailedRouting")

drt = DetailedRouting(state_in=grt.state_out)
drt.start()

In [ ]:
display(drt)

### Fill Insertion

Finally, as we're done placing all the essential cells, the only thing left to
do is fill in the gaps.

We prioritize the use of decap (decoupling capacitor) cells, which
further supports the power distribution network, but when there aren't any
small enough cells, we just use regular fill cells.

In [ ]:
FillInsertion = Step.factory.get("OpenROAD.FillInsertion")

fill = FillInsertion(state_in=drt.state_out)
fill.start()

In [ ]:
display(fill)

### Parasitics Extraction a.k.a. Resistance/Capacitance Extraction (RCX)

This step does not alter the design- rather, it computes the
[Parasitic elements](https://en.wikipedia.org/wiki/Parasitic_element_(electrical_networks))
of the circuit, which have an effect of timing, as we prepare to do the final
timing analysis.

The parasitic elements are saved in the **Standard Parasitics Exchange Format**,
or SPEF. LibreLane creates a SPEF file for each interconnect corner as described in
the [Corners and STA](https://librelane.readthedocs.io/en/latest/usage/corners_and_sta.html)
section of the documentation.

In [ ]:
RCX = Step.factory.get("OpenROAD.RCX")

rcx = RCX(state_in=fill.state_out)
rcx.start()

### Static Timing Analysis (Post-PnR)

STA is a process that verifies that a chip meets certain constraints on clock
and data timings to run at its rated clock speed. See [Corners and STA](https://librelane.readthedocs.io/en/latest/usage/corners_and_sta.html)
in the documentation for more info.

---

This step generates two kinds of files:
* `.lib`: Liberty™-compatible Library files. Can be used to do static timing
  analysis when creating a design with this design as a sub-macro.
* `.sdf`: Standard Delay Format. Can be used with certain simulation software
  to do *dynamic* timing analysis.

Unfortunately, the `.lib` files coming out of LibreLane right now are not super
reliable for timing purposes and are only provided for completeness.

When using LibreLane-created macros withing other designs, it is best to use the
macro's final netlist and extracted parasitics instead.

In [ ]:
STAPostPNR = Step.factory.get("OpenROAD.STAPostPNR")

sta_post_pnr = STAPostPNR(state_in=rcx.state_out)
sta_post_pnr.start()

### Stream-out

Stream-out is the process of converting the designs from the abstract formats
using during floorplanning, placement and routing into a concrete format called
GDSII (lit. Graphic Design System 2), which is the final file that is then sent
for fabrication.

In [ ]:
StreamOut = Step.factory.get("KLayout.StreamOut")

gds = StreamOut(state_in=sta_post_pnr.state_out)
gds.start()

In [ ]:
display(gds)

### Design Rule Checks (DRC)

DRC determines that the final layout does not violate any of the rules set by
the foundry to ensure the design is actually manufacturable- for example,
not enough space between two wires, *too much* space between tap cells, and so
on.

A design not passing DRC will typically be rejected by the foundry, who
also run DRC on their side.

In [ ]:
DRC = Step.factory.get("Magic.DRC")

drc = DRC(state_in=gds.state_out)
drc.start()

### SPICE Extraction for Layout vs. Schematic Check

This step tries to reconstruct a SPICE netlist from the GDSII file, so it can
later be used for the **Layout vs. Schematic** (LVS) check.

In [ ]:
SpiceExtraction = Step.factory.get("Magic.SpiceExtraction")

spx = SpiceExtraction(state_in=drc.state_out)
spx.start()

### Layout vs. Schematic (LVS)

A comparison between the final Verilog netlist (from PnR) and the final
SPICE netlist (extracted.)

This check effectively compares the physically implemented circuit to the final
Verilog netlist output by OpenROAD.

The idea is, if there are any disconnects, shorts or other mismatches in the
physical implementation that do not exist in the logical view of the design,
they would be caught at this step.

Common issues that result in LVS violations include:
* Lack of fill cells or tap cells in the design
* Two unrelated signals to be shorted, or a wire to be disconnected (most
  commonly seen with misconfigured PDN)

Chips with LVS errors are typically dead on arrival.

In [ ]:
LVS = Step.factory.get("Netgen.LVS")

lvs = LVS(state_in=spx.state_out)
lvs.start()

In [ ]:
pwd

In [ ]:
import glob
from google.colab import files

# 1) Locate the final GDSII (and other key artifacts) in the run directory.
run_dir = "/content/librelane_run"
gds_files = sorted(glob.glob(f"{run_dir}/**/*.gds", recursive=True))
print("Final GDSII file(s):")
for f in gds_files:
    print("  ", f)

# 2) Zip the whole run: GDS, netlists, reports, logs, DRC & LVS results.
!zip -r -q /content/top_module_results.zip {run_dir}
print("\nCreated /content/top_module_results.zip")

# 3) Download the zip to your computer.
files.download("/content/top_module_results.zip")
